# EDA — giao dịch bán lẻ điện tử

Notebook kiểm tra cấu trúc, chất lượng và mức tổng hợp của dữ liệu giao dịch. **Đề xuất bài toán, người dùng và cách đánh giá** được trình bày trong [README.md](README.md). Các ô cuối notebook tính số liệu kiểm chứng cho README.

> Chạy các ô từ trên xuống. File nguồn: `data/electronics-retail-transaction.xlsx`. Không chỉnh sửa file nguồn.

## A. Cấu trúc và chất lượng dữ liệu

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_excel('data/electronics-retail-transaction.xlsx', sheet_name='Data')
profile = pd.DataFrame({
    'kiểu đọc từ Excel': df.dtypes.astype(str),
    'unique': df.nunique(dropna=True),
    'thiếu': df.isna().sum(),
    'thiếu %': (100 * df.isna().mean()).round(1),
})
print(f'{len(df):,} dòng × {df.shape[1]} cột; trùng toàn dòng: {df.duplicated().sum():,}')
display(profile)
print('Lưu ý: object gồm chuỗi và cả ngày hỗn hợp; số hóa đơn dạng float do có ô trống.')

11,800 dòng × 48 cột; trùng toàn dòng: 0


,kiểu đọc từ Excel,unique,thiếu,thiếu %
ID,int64,11740,0,0.0
Đơn vị,object,1,0,0.0
Mã KVKD,object,1,0,0.0
Mã cửa hàng,object,11,0,0.0
Mã ca,int64,2,0,0.0
Loại hình bán,object,5,7,0.1
Ký hiệu HĐĐT,object,1,141,1.2
Số hóa đơn điện tử,float64,6329,148,1.3
Ngày hóa đơn ĐT,object,40,141,1.2
MST đơn vị mua hàng,float64,0,11800,100.0


Lưu ý: object gồm chuỗi và cả ngày hỗn hợp; số hóa đơn dạng float do có ô trống.


### Ngày: kiểm tra hai cách hiểu

`Ngày CT` có cả ngày Excel và chuỗi `DD/MM/YYYY`. Excel đã biến một số ngày 01–12/08 và 01–09/09 thành ngày có tháng/ngày đảo. Không dùng `pd.to_datetime(..., dayfirst=True)` trực tiếp cho cả cột. Quy tắc dưới đây được kiểm bằng cột `Ngày CT đ` và chuỗi 40 ngày liên tục.

In [2]:
from datetime import datetime

raw = df['Ngày CT']
is_excel_date = raw.map(lambda x: isinstance(x, (pd.Timestamp, datetime)))
assert raw.loc[is_excel_date].map(lambda x: x.day in (8, 9)).all()

date = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
date.loc[is_excel_date] = pd.to_datetime(raw.loc[is_excel_date].map(
    lambda x: x.replace(month=x.day, day=x.month)))
date.loc[~is_excel_date] = pd.to_datetime(raw.loc[~is_excel_date],
                                          format='%d/%m/%Y', errors='coerce')

shown = df['Ngày CT đ'].dropna().astype(str)
shown_date = pd.Series(pd.NaT, index=shown.index, dtype='datetime64[ns]')
dd_mask = shown.str.match(r'^\d{2}/08/2026$') & (shown.str[:2].astype(int) > 12)
shown_date.loc[dd_mask] = pd.to_datetime(shown.loc[dd_mask], format='%d/%m/%Y')
shown_date.loc[~dd_mask] = pd.to_datetime(shown.loc[~dd_mask], format='%m/%d/%Y')

assert date.notna().all() and (shown_date == date.loc[shown.index]).all()
df['ngay_dung'] = date
print(f'Ngày Excel: {is_excel_date.sum():,}; chuỗi DD/MM/YYYY: {(~is_excel_date).sum():,}')
print(f'Khoảng thực tế: {date.min():%d/%m/%Y} – {date.max():%d/%m/%Y}; '
      f'{date.nunique()} ngày giao dịch / {(date.max()-date.min()).days+1} ngày liên tiếp')
print(f'Đối chiếu Ngày CT đ: {(shown_date == date.loc[shown.index]).sum():,}/{len(shown):,} dòng khớp')
print('Nếu parse sai sẽ thấy 08/01–08/12/2026 và tưởng có gần một năm dữ liệu.')

Ngày Excel: 6,095; chuỗi DD/MM/YYYY: 5,705
Khoảng thực tế: 01/08/2026 – 09/09/2026; 40 ngày giao dịch / 40 ngày liên tiếp
Đối chiếu Ngày CT đ: 2,901/2,901 dòng khớp
Nếu parse sai sẽ thấy 08/01–08/12/2026 và tưởng có gần một năm dữ liệu.


**Thiếu có ý nghĩa:** MST và tên đơn vị mua hàng trống 100%; 7 cột tiền phí/MBF luôn bằng 0; mã đơn hàng và vận đơn trống khoảng 95%. Không tự điền các trường này thành thông tin có thật.

## B. Đơn vị quan sát và khóa

In [3]:
order = 'Stt đơn hàng'
invoice = 'Số hóa đơn điện tử'
order_size = df.groupby(order).size()
valid_invoice = df[df[invoice].notna()]
checks = pd.Series({
    'Dòng': len(df),
    'ID unique': df['ID'].nunique(),
    'ID bị lặp (giá trị)': (df['ID'].value_counts() > 1).sum(),
    'Đơn unique': df[order].nunique(),
    'Hóa đơn có số': valid_invoice[invoice].nunique(),
    'Đơn có nhiều dòng': (order_size > 1).sum(),
    'Dòng/đơn tối đa': order_size.max(),
    'Đơn thiếu số hóa đơn': df.loc[df[invoice].isna(), order].nunique(),
    'Đơn trải nhiều cửa hàng': (df.groupby(order)['Mã cửa hàng'].nunique() > 1).sum(),
    'Đơn trải nhiều ngày': (df.groupby(order)['ngay_dung'].nunique() > 1).sum(),
    'Đơn có nhiều số hóa đơn': (df.groupby(order)[invoice].nunique() > 1).sum(),
    'Số hóa đơn gắn nhiều đơn': (valid_invoice.groupby(invoice)[order].nunique() > 1).sum(),
    'Cặp đơn–SKU lặp': df.duplicated([order, 'Mã vật tư']).sum(),
})
display(checks.to_frame('giá trị'))
print('Phân bố số dòng/đơn:')
display(order_size.value_counts().sort_index().to_frame('số đơn').T)
print('=> Khóa đơn thực dụng: Stt đơn hàng. Một dòng = một line item; ID không phải khóa duy nhất.')

,giá trị
Dòng,11800
ID unique,11740
ID bị lặp (giá trị),60
Đơn unique,6453
Hóa đơn có số,6329
Đơn có nhiều dòng,3011
Dòng/đơn tối đa,25
Đơn thiếu số hóa đơn,124
Đơn trải nhiều cửa hàng,0
Đơn trải nhiều ngày,0


Phân bố số dòng/đơn:


,1,2,3,4,5,6,7,8,9,10,17,25
số đơn,3442,1784,588,338,218,54,13,9,2,2,2,1


=> Khóa đơn thực dụng: Stt đơn hàng. Một dòng = một line item; ID không phải khóa duy nhất.


## C. Thực thể, cửa hàng và mức tổng hợp

In [4]:
entities = pd.Series({
    'Cửa hàng': df['Mã cửa hàng'].nunique(),
    'Khu vực': df['Mã KVKD'].nunique(),
    'Nhân viên': df['Mã NVBH'].nunique(),
    'Tên khách hàng (không phải customer ID)': df['Tên khách hàng'].nunique(),
    'Đơn': df[order].nunique(),
    'Hóa đơn có số': valid_invoice[invoice].nunique(),
    'SKU': df['Mã vật tư'].nunique(),
    'Hãng': df['Hãng SX'].nunique(),
    'Chủng loại': df['Chủng loại'].nunique(),
    'Nhóm hàng (mã)': df['Loại nhóm hàng'].nunique(),
})
display(entities.to_frame('số unique'))
store = df.groupby('Mã cửa hàng').agg(
    dong=('ID', 'size'), don=(order, 'nunique'),
    hoa_don=(invoice, 'nunique'), doanh_thu=('Tổng đơn KH', 'sum'))
display(store)
print('Có thể tổng hợp: line → đơn/giỏ → cửa hàng × ngày → SKU/nhóm × cửa hàng × ngày.')
print('Không có nhiều khu vực để học khác biệt giữa khu vực.')

,số unique
Cửa hàng,11
Khu vực,1
Nhân viên,81
Tên khách hàng (không phải customer ID),3791
Đơn,6453
Hóa đơn có số,6329
SKU,1507
Hãng,146
Chủng loại,21
Nhóm hàng (mã),757


,dong,don,hoa_don,doanh_thu
Mã cửa hàng,,,,
HN006,2305,1297,1269,10808502000
HN009,1556,794,789,5583034000
HN011,1064,587,580,3456633000
HN015,427,218,215,1202084000
HN016,464,250,250,1632499000
HN017,669,393,382,3056915000
HN018,804,474,464,3045749000
HN020,807,421,414,2640225000
HN021,1168,621,605,4330904000


Có thể tổng hợp: line → đơn/giỏ → cửa hàng × ngày → SKU/nhóm × cửa hàng × ngày.
Không có nhiều khu vực để học khác biệt giữa khu vực.


## D. Thông tin độc lập, trùng lặp và rò rỉ

In [5]:
sku_fields = ['Tên vật tư', 'Chủng loại', 'Hãng SX', 'Loại nhóm hàng', 'Đvt']
sku_conflicts = {c: (df.groupby('Mã vật tư')[c].nunique(dropna=True) > 1).sum()
                 for c in sku_fields}
print('Số SKU ánh xạ sang >1 giá trị ở từng trường:', sku_conflicts)
print('Mã NVBH → >1 tên:', (df.groupby('Mã NVBH')['Tên NVBH'].nunique() > 1).sum())
print('Giá bán KH = Doanh thu KH:', (df['Giá bán KH'] == df['Doanh thu KH']).sum(), '/', len(df))
print('Doanh thu KH + Thuế KH = Tổng đơn KH:',
      ((df['Doanh thu KH'] + df['Thuế KH']) == df['Tổng đơn KH']).sum(), '/', len(df))
print('Số lượng = 1:', (df['Số lượng'] == 1).sum(), '; = 0:', (df['Số lượng'] == 0).sum())
print('Doanh thu dòng = 0:', (df['Tổng đơn KH'] == 0).sum())
print('Doanh thu âm:', (df['Tổng đơn KH'] < 0).sum())

Số SKU ánh xạ sang >1 giá trị ở từng trường: {'Tên vật tư': np.int64(0), 'Chủng loại': np.int64(0), 'Hãng SX': np.int64(0), 'Loại nhóm hàng': np.int64(0), 'Đvt': np.int64(0)}


Mã NVBH → >1 tên: 0
Giá bán KH = Doanh thu KH: 11785 / 11800
Doanh thu KH + Thuế KH = Tổng đơn KH: 11800 / 11800
Số lượng = 1: 11785 ; = 0: 15
Doanh thu dòng = 0: 890
Doanh thu âm: 0


| Nhóm | Cột và quyết định |
|---|---|
| Khóa, nhãn | `ID`, `Stt đơn hàng`, số hóa đơn, mã đơn hàng, vận đơn: dùng để nối/gom; bỏ khỏi feature số. Tên khách hàng không định danh khách ổn định. |
| Gần như hằng/trống | Đơn vị, Mã KVKD, ký hiệu HĐĐT, MST/tên đơn vị mua, doanh thu/thuế phí, toàn bộ MBF: bỏ. |
| Tra cứu trùng | Tên SKU, chủng loại, hãng, nhóm và Đvt xác định theo mã SKU trong mẫu; chọn cấp phù hợp, tránh đưa đồng thời tất cả dưới dạng one-hot. Tên NVBH đi theo mã NVBH. |
| Kết quả sau bán | Doanh thu, thuế, tổng đơn, hoa hồng, giảm giá, hóa đơn, ngày lập/sửa: rò rỉ nếu dự đoán trước lúc bán. Với dự báo ngày kế tiếp, chỉ dùng các giá trị đã biết trước ngày mục tiêu. |
| Cần kiểm tra nghiệp vụ | `Mã kho`, `Tình trạng`, `Ghi chú`, `Mã hạng`, đối tác: có thể mang tín hiệu, nhưng nhiều ô trống/ghi chú tự do; chỉ dùng khi biết thời điểm phát sinh và ý nghĩa. |

**Lưu ý:** mã SKU xác định các nhãn danh mục trong *mẫu hiện tại*; không khẳng định quy tắc đó luôn đúng cho dữ liệu tương lai.

## E. Phân bố, giỏ hàng và giới hạn mẫu

In [6]:
basket = df.groupby(order).agg(
    ngay=('ngay_dung', 'first'), cua_hang=('Mã cửa hàng', 'first'),
    so_dong=('ID', 'size'), so_sku=('Mã vật tư', 'nunique'),
    gia_tri=('Tổng đơn KH', 'sum'))
print('Giá trị đơn (VND):')
display(basket[['so_dong', 'so_sku', 'gia_tri']].describe(percentiles=[.5,.75,.9,.95,.99]).round(0))
print('Loại hình bán (% dòng):')
display((df['Loại hình bán'].value_counts(dropna=False, normalize=True)*100).round(1))
print('Đơn không có doanh thu:', (basket.gia_tri == 0).sum())
print('Tên khách hàng lặp ở >1 đơn:',
      (df.groupby('Tên khách hàng')[order].nunique() > 1).sum(),
      '— có thể trùng tên người khác nhau, không xem là khách quay lại thật.')

Giá trị đơn (VND):


,so_dong,so_sku,gia_tri
count,6453.0,6453.0,6453.0
mean,2.0,2.0,7297954.0
std,1.0,1.0,12799136.0
min,1.0,1.0,0.0
50%,1.0,1.0,3280000.0
75%,2.0,2.0,7747000.0
90%,3.0,3.0,23888800.0
95%,4.0,4.0,34191600.0
99%,6.0,5.0,41471280.0
max,25.0,8.0,437250000.0


Loại hình bán (% dòng):


Loại hình bán
Bán lẻ            97.4
Thu cũ đổi mới     1.4
Bán dịch vụ        0.9
Bán buôn           0.2
Trả nợ quà         0.1
NaN                0.1
Name: proportion, dtype: float64

Đơn không có doanh thu: 10
Tên khách hàng lặp ở >1 đơn: 754 — có thể trùng tên người khác nhau, không xem là khách quay lại thật.


## F. Bảng phân tích sẵn cho Data Mining/ML

In [7]:
# Giỏ hàng: một đơn có thể chứa nhiều SKU; phù hợp Apriori/FP-Growth.
order_sku = (df.groupby([order, 'Mã vật tư']).size()
             .unstack(fill_value=0).gt(0))

# Mở rộng đủ lịch để shift(1) và shift(-1) luôn là ngày lịch liền kề.
# Ô thiếu để NaN: không biết cửa hàng đóng cửa hay dữ liệu bị thiếu, nên không tự gán 0.
stores = df['Mã cửa hàng'].drop_duplicates().sort_values()
dates = pd.date_range(df['ngay_dung'].min(), df['ngay_dung'].max(), freq='D')
calendar = pd.MultiIndex.from_product([stores, dates], names=['Mã cửa hàng', 'ngay_dung'])
daily = df.groupby(['Mã cửa hàng', 'ngay_dung']).agg(
    doanh_thu=('Tổng đơn KH', 'sum'), so_don=(order, 'nunique'),
    so_dong=('ID', 'size')).reindex(calendar)
g = daily.groupby(level=0)
daily['doanh_thu_t_1'] = g['doanh_thu'].shift(1)
daily['doanh_thu_t_7'] = g['doanh_thu'].shift(7)
daily['tb_7_ngay_truoc'] = g['doanh_thu'].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean())
daily['doanh_thu_ngay_sau'] = g['doanh_thu'].shift(-1)
forecast = daily.dropna().copy()
print('Ma trận giỏ hàng (đơn × SKU):', order_sku.shape)
print('Bảng cửa hàng × ngày:', daily.shape, '; ô cửa hàng/ngày thiếu:', daily.so_don.isna().sum(),
      '; mẫu đủ lag và target:', forecast.shape)
display(forecast.head(3))

Ma trận giỏ hàng (đơn × SKU):

 (6453, 1507)
Bảng cửa hàng × ngày: (440, 7) ; ô cửa hàng/ngày thiếu: 1 ; mẫu đủ lag và target: (343, 7)


doanh_thu  so_don  so_dong  doanh_thu_t_1  \
Mã cửa hàng ngay_dung                                                 
HN006       2026-08-08  569432000.0    38.0     66.0    253854000.0   
            2026-08-09  259546000.0    48.0     85.0    569432000.0   
            2026-08-10  283735000.0    24.0     37.0    259546000.0   

                        doanh_thu_t_7  tb_7_ngay_truoc  doanh_thu_ngay_sau  
Mã cửa hàng ngay_dung                                                       
HN006       2026-08-08    391526000.0     2.858349e+08         259546000.0  
            2026-08-09    458652000.0     3.112500e+08         283735000.0  
            2026-08-10    163326000.0     2.828063e+08         240017000.0

## G. Số liệu kiểm chứng cho các bài toán trong README

### G1. Độ phủ giao dịch theo cửa hàng và ca

In [8]:
retail = df[df['Loại hình bán'] == 'Bán lẻ']
shifts = df['Mã ca'].drop_duplicates().sort_values()
shift_calendar = pd.MultiIndex.from_product(
    [stores, shifts, dates], names=['Mã cửa hàng', 'Mã ca', 'ngay_dung'])
shift_orders = (retail.groupby(['Mã cửa hàng', 'Mã ca', 'ngay_dung'])[order]
                .nunique().reindex(shift_calendar).rename('so_don_ban_le'))
print('Cửa hàng × ca × ngày có đơn bán lẻ:', shift_orders.notna().sum(), '/', len(shift_orders))
print('Ô chưa rõ đóng ca/không bán/thiếu dữ liệu:', shift_orders.isna().sum())
print('Số đơn theo ca (chỉ ô quan sát được):')
display(shift_orders.groupby(level='Mã ca').describe(percentiles=[.5,.9]).round(1))
print('Mỗi đơn có nhiều mã ca:', (df.groupby(order)['Mã ca'].nunique() > 1).sum())

Cửa hàng × ca × ngày có đơn bán lẻ:

 845 / 880
Ô chưa rõ đóng ca/không bán/thiếu dữ liệu: 35
Số đơn theo ca (chỉ ô quan sát được):


,count,mean,std,min,50%,90%,max
Mã ca,,,,,,,
1,438.0,10.0,6.7,1.0,8.0,19.0,39.0
2,407.0,4.6,3.2,1.0,4.0,9.0,20.0


Mỗi đơn có nhiều mã ca: 0


In [9]:
by_store_shift = shift_orders.groupby(level=['Mã cửa hàng','Mã ca'])
pred_yesterday = by_store_shift.shift(1)
pred_avg7 = by_store_shift.transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean())
test_day = shift_orders.index.get_level_values('ngay_dung') >= pd.Timestamp('2026-09-03')
comparison = pd.DataFrame({
    'thực tế': shift_orders,
    'hôm qua': pred_yesterday,
    'trung bình 7 ngày trước': pred_avg7,
}).loc[test_day].dropna()
mae = comparison[['hôm qua', 'trung bình 7 ngày trước']].sub(
    comparison['thực tế'], axis=0).abs().mean()
print('Mẫu kiểm tra đủ dữ liệu:', len(comparison), '/', test_day.sum())
print('Số đơn thực tế trung bình:', round(comparison['thực tế'].mean(), 2))
display(mae.rename('MAE (đơn/cửa hàng-ca-ngày)').round(2).to_frame())
print('Theo ca — MAE của mốc trung bình 7 ngày:')
display(comparison.assign(sai_so=(comparison['trung bình 7 ngày trước']-comparison['thực tế']).abs())
        .groupby(level='Mã ca')['sai_so'].mean().round(2))

Mẫu kiểm tra đủ dữ liệu: 118 / 154
Số đơn thực tế trung bình: 8.19


,MAE (đơn/cửa hàng-ca-ngày)
hôm qua,3.72
trung bình 7 ngày trước,3.21


Theo ca — MAE của mốc trung bình 7 ngày:


Mã ca
1    3.72
2    2.43
Name: sai_so, dtype: float64

### G2. Độ phủ lượng bán điện thoại và phụ kiện

In [10]:
retail_positive = df[(df['Loại hình bán'] == 'Bán lẻ') &
                     (df['Tổng đơn KH'] > 0) & (df['Số lượng'] > 0)]
core_types = ['DIENTHOAI', 'PHUKIEN']
cat_calendar = pd.MultiIndex.from_product(
    [stores, core_types, dates], names=['Mã cửa hàng', 'Chủng loại', 'ngay_dung'])
cat_units = (retail_positive[retail_positive['Chủng loại'].isin(core_types)]
             .groupby(['Mã cửa hàng', 'Chủng loại', 'ngay_dung'])['Số lượng']
             .sum().reindex(cat_calendar).rename('so_luong_da_ban'))
print('Cửa hàng × ngày × nhóm có bán:', cat_units.notna().sum(), '/', len(cat_units))
display(cat_units.groupby(level='Chủng loại').agg(['count','median','min','max']))
cat_g = cat_units.groupby(level=['Mã cửa hàng','Chủng loại'])
cat_test = cat_units.index.get_level_values('ngay_dung') >= pd.Timestamp('2026-09-03')
cat_eval = pd.DataFrame({
    'thực tế': cat_units,
    'hôm qua': cat_g.shift(1),
    'trung bình 7 ngày trước': cat_g.transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).mean()),
}).loc[cat_test].dropna()
print('Mẫu kiểm tra đủ lịch sử:', len(cat_eval), '/', cat_test.sum())
display(cat_eval.assign(sai_so_hom_qua=(cat_eval['hôm qua']-cat_eval['thực tế']).abs(),
                        sai_so_tb7=(cat_eval['trung bình 7 ngày trước']-cat_eval['thực tế']).abs())
        .groupby(level='Chủng loại')[['thực tế','sai_so_hom_qua','sai_so_tb7']]
        .mean().round(2))

Cửa hàng × ngày × nhóm có bán: 868 / 880


,count,median,min,max
Chủng loại,,,,
DIENTHOAI,435,6.0,1.0,26.0
PHUKIEN,433,9.0,1.0,41.0


Mẫu kiểm tra đủ lịch sử: 147 / 154


,thực tế,sai_so_hom_qua,sai_so_tb7
Chủng loại,,,
DIENTHOAI,5.61,3.05,2.79
PHUKIEN,9.59,4.76,4.67


### G3. Cấu trúc giỏ hàng bán lẻ

In [11]:
basket_types = retail_positive.groupby(order)['Chủng loại'].nunique()
basket_skus = retail_positive.groupby(order)['Mã vật tư'].nunique()
print('Giỏ bán lẻ có doanh thu dương:', len(basket_types))
display(basket_types.value_counts().sort_index().rename('số giỏ').to_frame())
print('Giỏ có từ 2 chủng loại:', (basket_types >= 2).sum(),
      f'({(basket_types >= 2).mean():.1%})')
print('Giỏ có từ 2 SKU:', (basket_skus >= 2).sum(),
      f'({(basket_skus >= 2).mean():.1%})')

Giỏ bán lẻ có doanh thu dương: 6262


,số giỏ
Chủng loại,
1,4043
2,1689
3,513
4,15
5,2


Giỏ có từ 2 chủng loại: 2219 (35.4%)
Giỏ có từ 2 SKU: 2528 (40.4%)


### G4. Liên hệ mua cùng giữa các nhóm hàng

In [12]:
category_sets = retail_positive.groupby(order)['Chủng loại'].agg(lambda s: set(s))
has_phone = category_sets.map(lambda s: 'DIENTHOAI' in s)
pair_rows = []
for item in ['PHUKIEN', 'HCARE001']:
    has_item = category_sets.map(lambda s: item in s)
    both = (has_phone & has_item).sum()
    pair_rows.append({
        'cặp với DIENTHOAI': item,
        'đơn cùng xuất hiện': both,
        'support': both / len(category_sets),
        'confidence': both / has_phone.sum(),
        'lift': (both / has_phone.sum()) / has_item.mean(),
    })
display(pd.DataFrame(pair_rows).set_index('cặp với DIENTHOAI').round(3))

,đơn cùng xuất hiện,support,confidence,lift
cặp với DIENTHOAI,,,,
PHUKIEN,1276,0.204,0.453,0.902
HCARE001,775,0.124,0.275,1.505


### G5. Đơn chưa có số hóa đơn

In [13]:
missing_invoice = df[df[invoice].isna()]
print('Dòng thiếu số hóa đơn:', len(missing_invoice),
      '| đơn khác nhau:', missing_invoice[order].nunique())
display(missing_invoice.groupby('Loại hình bán', dropna=False)[order]
        .nunique().rename('số đơn thiếu số hóa đơn').to_frame())
print('Giá trị đơn: trung vị và lớn nhất (VND):',
      basket['gia_tri'].median(), basket['gia_tri'].max())

Dòng thiếu số hóa đơn: 148 | đơn khác nhau: 124


,số đơn thiếu số hóa đơn
Loại hình bán,
Bán buôn,6
Bán dịch vụ,103
Bán lẻ,5
Trả nợ quà,10


Giá trị đơn: trung vị và lớn nhất (VND): 3280000.0 437250000
